In [3]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import sys, os
import datetime
sys.path.append('/home/548/cd3022/repos/Irradiance-comparisons/Irradiance-comparisons')
from read_datasets import read_dataset
import logger

LOG = logger.get_logger(__name__)

from dask.distributed import Client

In [4]:
client = Client(
    n_workers=6,
    threads_per_worker=1
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 6
Total threads: 6,Total memory: 95.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41601,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33031,Total threads: 1
Dashboard: /proxy/33119/status,Memory: 15.83 GiB
Nanny: tcp://127.0.0.1:39899,


In [5]:
dataset = 'himawari'
year = 2019
LOG.info(f'Starting analysis for dataset {dataset}, sea breezes, year {year}')

# GHI dataset
ds_list = []
for month in range(1, 2):
    # LOAD DATASETS
    ds_month = read_dataset(
            dataset=dataset,
            resolution='hourly',
            date=f'{year}-{month:02d}'
        )
    ds_list.append(ds_month)
ds = xr.concat(ds_list, dim='time')
LOG.info(f'{dataset} data opened')

sb_path = Path('/g/data/ng72/ab4502/sea_breeze_detection/barra_c_smooth_s2/filters/')
sb_files = [f for f in sb_path.glob(f'*_F_{year}01*.zarr')]
ds_sb = xr.open_mfdataset(
    sb_files,
    engine="zarr",
    combine="by_coords",
    compat='override',
    coords='minimal',
    parallel=True,
)
LOG.info('sea breeze data opened')

2026-01-30 16:56:48,788:__main__:INFO: Starting analysis for dataset himawari, sea breezes, year 2019
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backe

In [6]:
# PREPROCESS DATASETS TO MATCH TIMES/SHAPES
# interp wf mask from era5 to barra
ds_sb = ds_sb.interp(
    lat=ds.lat,
    lon=ds.lon,
    method='nearest'
)

# adjust hourly times on the 30min to match 3hr times (on the hour) from wf
ds_shifted = ds.assign_coords(time=ds.time + pd.Timedelta('30min'))

if dataset == 'himawari':
    # fill missing overnight time steps
    full_time = pd.date_range(
        start=ds_shifted.time.min().item(),
        end=ds_shifted.time.max().item(),
        freq="60min"
    )
    ds_shifted = ds_shifted.reindex(time=full_time, fill_value=0)
    
    # line up datasets
    min_time = ds_sb.time.min()
    max_time = ds_shifted.time.max()
    t_range = slice(min_time, max_time)
    ds_sb = ds_sb.sel(time=t_range)
    ds = ds.sel(time=t_range)

ds_times = ds_shifted.sel(time=ds_sb.time)
LOG.info('preprocessing complete')

# Extend mask to capture larger area round sea breeze
mask = ds_sb.mask.fillna(False).astype(bool)
sb_extension = 2 # hours
sb_extended = (
    mask
    | mask.shift(time=sb_extension).fillna(False)
    | mask.shift(time=-sb_extension).fillna(False)
)
LOG.info(f'sb mask extended forward and back {sb_extension} hours')

2026-01-30 16:58:53,339:__main__:INFO: preprocessing complete
2026-01-30 16:58:53,366:__main__:INFO: sb mask extended forward and back 2 hours


In [8]:
# APPLY MASK
sb_ghi = xr.where(sb_extended != 0, ds_times.ghi, np.nan)
LOG.info('mask applied')

# TIME MEAN
sb_ghi_mean = sb_ghi.mean(dim='time')
LOG.info('annual mean calculated')

# remask himawari region
if dataset == 'himawari':
    sb_ghi_mean = xr.where(ds.isel(time=5).ghi.isnull(), np.nan, sb_ghi_mean)

2026-01-30 16:59:30,874:__main__:INFO: mask applied
2026-01-30 16:59:30,895:__main__:INFO: annual mean calculated


In [ ]:
sb_ghi_mean.plot()

In [ ]:
# add metadata
sb_ghi_mean = sb_ghi_mean.to_dataset(name='annual_mean_ghi')
sb_ghi_mean = sb_ghi_mean.assign_coords({'year':year})
sb_ghi_mean.attrs['date_generated'] = datetime.date.today().strftime('%D')
sb_ghi_mean.attrs['source_script'] = 'data produced by the script "041.5_sea_breezes.py"'

# SAVE DATA
file_path = Path(f'/g/data/er8/users/cd3022/Irradiance-comparisons/weather-features/sea_breaze')
os.makedirs(file_path, exist_ok=True)
sb_ghi_mean.to_netcdf(f'{file_path}/{dataset}-{year}.nc')
LOG.info('data saved, job complete!')